# Precision Comparison Experiment
This notebook compares FP32, FP16, and AMP for medical image classification.

In [ ]:
import sys
import os
sys.path.append('.')
from src.dataset import get_dataloaders
from src.model import get_model
from src.train import train_model
from src.utils import set_seed
import matplotlib.pyplot as plt
import pandas as pd
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

EPOCHS = 5
BATCH_SIZE = 32
LR = 1e-4

results = {}

In [ ]:
# FP32 Training
set_seed(42)
train_loader, val_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model_fp32 = get_model(num_classes=2, pretrained=True)
print("\n--- Starting FP32 Training ---")
model_fp32, history_fp32, peak_mem_fp32 = train_model(
    model_fp32, train_loader, val_loader, 
    precision='fp32', epochs=EPOCHS, lr=LR, device=device
)
results['fp32'] = {'history': history_fp32, 'peak_memory': peak_mem_fp32}

In [ ]:
# FP16 (Pure) Training
set_seed(42)
model_fp16 = get_model(num_classes=2, pretrained=True)
print("\n--- Starting Pure FP16 Training ---")
model_fp16, history_fp16, peak_mem_fp16 = train_model(
    model_fp16, train_loader, val_loader, 
    precision='fp16', epochs=EPOCHS, lr=LR, device=device
)
results['fp16'] = {'history': history_fp16, 'peak_memory': peak_mem_fp16}

In [ ]:
# AMP Training
set_seed(42)
model_amp = get_model(num_classes=2, pretrained=True)
print("\n--- Starting AMP Training ---")
model_amp, history_amp, peak_mem_amp = train_model(
    model_amp, train_loader, val_loader, 
    precision='amp', epochs=EPOCHS, lr=LR, device=device
)
results['amp'] = {'history': history_amp, 'peak_memory': peak_mem_amp}

In [ ]:
# Visualization and Results
summary = []
for precision in ['fp32', 'fp16', 'amp']:
    h = results[precision]['history']
    summary.append({
        'Precision': precision.upper(),
        'Avg Time/Epoch (s)': sum(h['epoch_times'])/len(h['epoch_times']),
        'Total Time (s)': sum(h['epoch_times']),
        'Peak Memory (MB)': results[precision]['peak_memory'],
        'Final Val Acc': h['val_acc'][-1],
        'Final Val AUC': h['val_auc'][-1],
        'Final Val F1': h['val_f1'][-1]
    })

df = pd.DataFrame(summary)
display(df)
df.to_csv('results_summary.csv', index=False)

plt.figure(figsize=(10, 5))
for p in ['fp32', 'fp16', 'amp']:
    plt.plot(results[p]['history']['train_loss'], label=p.upper())
plt.title('Training Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('train_loss.png')
plt.show()

plt.figure(figsize=(10, 5))
for p in ['fp32', 'fp16', 'amp']:
    plt.plot(results[p]['history']['val_auc'], label=p.upper())
plt.title('Validation AUC-ROC per Epoch')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()
plt.savefig('val_auc.png')
plt.show()